# 02. Explore SteamIDX and SQLite

Inspect API responses, compare Steam App IDs with FronkonGames, and test database persistence.

## 1. Setup

In [ ]:
import csv
import json
import sqlite3
from pprint import pprint

import numpy as np
import pandas as pd

from video_game_retention.api import (
    get_steam_game_details,
    get_steam_player_history,
    get_top_games,
)
from video_game_retention.config import DATABASE_PATH, PROJECT_ROOT
from video_game_retention.database import init_database

## 2. Explore the SteamIDX API

Fetch game details and player history to inspect the response structure, data types, and available observation dates.

In [ ]:
# Grand Theft Auto V Legacy (271590)
game = get_steam_game_details(271590)
history = get_steam_player_history(271590)

points = history["points"]

print("Game:", game["name"])
print("Release date:", game["releaseDate"])
print("History resolution:", history["resolution"])
print("Observations:", len(points))

if points:
    print("Earliest observation:", min(p["at"] for p in points))
    print("Latest observation:", max(p["at"] for p in points))

In [ ]:
# variable type
for key, value in game.items():
    print(f"{key}: {type(value).__name__}")

In [ ]:
pprint(game)

In [ ]:
app_ids = [
    730,      # Counter-Strike 2
    1086940,  # Baldur's Gate 3
    1091500,  # Cyberpunk 2077
    271590,
]

results = []

for app_id in app_ids:
    game = get_steam_game_details(app_id)
    history = get_steam_player_history(app_id, "all")

    points = history.get("points", [])

    results.append({
        "app_id": app_id,
        "name": game["name"],
        "release_date": game["releaseDate"],
        "first_observation": min(
            (p["at"] for p in points),
            default=None,
        ),
        "last_observation": max(
            (p["at"] for p in points),
            default=None,
        ),
        "observations": len(points),
        "resolution": history.get("resolution"),
    })

df = pd.DataFrame(results)
display(df)

The earliest observations in this sample appear relatively recent. Limited historical coverage may constrain the retention analysis; this needs to be assessed across a larger sample.

## 3. Compare SteamIDX with FronkonGames

Check whether Steam App IDs from the top 100 SteamIDX games are present in the corrected FronkonGames dataset.

In [ ]:
top_games = get_top_games(100)

print("Response type:", type(top_games).__name__)
print("Games retrieved:", len(top_games))

In [ ]:
pprint(top_games[:10])

In [ ]:
# Load FronkonGames using the header correction from notebook 01 
csv_path = f"{PROJECT_ROOT}/data/raw/games.csv"

with open(csv_path, encoding="utf-8", newline="") as f:
    header = next(csv.reader(f))

corrected_header = (
    header[:7]
    + ["Discount", "DLC count"]
    + header[8:]
)

fronkon_df = pd.read_csv(
    csv_path,
    header=0,
    names=corrected_header,
    index_col=False
)

fronkon_ids = (
    pd.to_numeric(fronkon_df["AppID"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .astype(int)
    .drop_duplicates()
    .tolist()
)

In [ ]:
top_df = pd.DataFrame(top_games)

top_df["in_fronkon"] = top_df["appId"].isin(fronkon_ids)

print("Already in FronkonGames:", top_df["in_fronkon"].sum())
print("Missing from FronkonGames:", (~top_df["in_fronkon"]).sum())

In [ ]:
# GTA V legacy in?
print("In FronkonGames:", 271590 in fronkon_ids)

In [ ]:
print("In SteamIDX top 100:", 271590 in top_df["appId"].astype(int).tolist())

In [ ]:
top_df.sort_values(by="allTimePeak", ascending=False).head()

In [ ]:
top_100_id = top_df.sort_values(by="allTimePeak", ascending=False)["appId"].tolist()

In [ ]:
top_100_id

In [ ]:
top_df[~top_df["in_fronkon"]][
    ["appId", "name"]
]

## 4. Test the SQLite database

Initialize the database and inspect its tables. The following tests write to the configured database.

In [ ]:
init_database()
print(DATABASE_PATH)

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        conn,
    )

tables

### 4.1. Test a manual insert into `games`

Fetch Portal 2 (App ID 620), upsert its metadata, and read it back to verify storage.

In [ ]:
game = get_steam_game_details(620)

with sqlite3.connect(DATABASE_PATH) as conn:
    conn.execute(
        """
        INSERT INTO games (app_id, name, details_json)
        VALUES (?, ?, ?)
        ON CONFLICT(app_id) DO UPDATE SET
            name = excluded.name,
            details_json = excluded.details_json,
            collected_at = CURRENT_TIMESTAMP
        """,
        (
            game["appId"],
            game["name"],
            json.dumps(game),
        ),
    )

print("Saved:", game["name"])

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    saved_game = pd.read_sql_query(
        """
        SELECT app_id, name, collected_at, details_json
        FROM games
        WHERE app_id = ?
        """,
        conn,
        params=(620,),
    )

saved_game

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    row = conn.execute(
        "SELECT details_json FROM games WHERE app_id = ?",
        (620,),
    ).fetchone()

saved_details = json.loads(row[0])

print(saved_details["name"])
print(saved_details["genres"])
print(saved_details["reviews"])

### 4.2. Test a manual insert into `player_history`

Fetch Portal 2 player history and upsert observations using the App ID and observation timestamp.

In [ ]:
history = get_steam_player_history(620, range_="all")

In [ ]:
print(type(history))
print(history if isinstance(history, dict) else history[:2])

In [ ]:
rows = [
    (history["appId"], point["at"], point["players"])
    for point in history["points"]
]

with sqlite3.connect(DATABASE_PATH) as conn:
    conn.executemany(
        """
        INSERT INTO player_history (app_id, observed_at, players)
        VALUES (?, ?, ?)
        ON CONFLICT(app_id, observed_at) DO UPDATE SET
            players = excluded.players
        """,
        rows,
    )

print(f"Saved {len(rows)} player-history observations.")

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    saved_history = pd.read_sql_query(
        """
        SELECT app_id, observed_at, players
        FROM player_history
        WHERE app_id = ?
        ORDER BY observed_at
        """,
        conn,
        params=(620,),
    )

saved_history.head()

In [ ]:
saved_history.tail()

In [ ]:
print("API observations:", len(history["points"]))
print("Database observations:", len(saved_history))

The manual insert above tests the same storage workflow intended for the `save_player_history(history)` helper. No additional write is performed here.

## 5. Check top-games retrieval

Request 200 games and inspect entries around rank 100.

In [ ]:
top_games = get_top_games(200)

print("Games retrieved:", len(top_games))
print("Unique App IDs:", len({game["appId"] for game in top_games}))

In [ ]:
for rank, game in enumerate(top_games[95:105], start=96):
    print(rank, game["appId"], game["name"])

## 6. Inspect database totals

Count stored game records and player-history observations.

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    result = conn.execute(
        "SELECT COUNT(*) FROM games"
    ).fetchone()

print(f"Total games in database: {result[0]}")

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    result = conn.execute("SELECT COUNT(*) FROM player_history").fetchone()

print(f"Total player-history observations in database: {result[0]}")